# Explore Parsed Workout Data

This notebook helps you explore and clean the workout data extracted from your OCR PDFs.

In [ ]:
import pandas as pd
import json
from pathlib import Path

# Load the parsed workout data
df = pd.read_csv('../data_samples/parsed_workouts.csv', parse_dates=['date'])
print(f"Total rows: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique dates (sessions): {df['date'].nunique()}")
df.head(10)

## Data Quality Overview

In [ ]:
# Check data completeness
print("=== Data Completeness ===")
print(f"Rows with weight data: {df['weight'].notna().sum()} ({df['weight'].notna().mean()*100:.1f}%)")
print(f"Rows with reps data: {df['sets'].notna().sum()} ({df['sets'].notna().mean()*100:.1f}%)")
print(f"Rows with both: {((df['weight'].notna()) & (df['sets'].notna())).sum()}")

## Clean Up the Data

Remove rows that aren't real exercises (dates, notes, etc.)

In [ ]:
# Remove non-exercise rows
df_clean = df.copy()

# Remove date lines captured as exercises
df_clean = df_clean[~df_clean['exercise'].str.contains(r'^Date\s*:', regex=True, na=False)]
df_clean = df_clean[~df_clean['exercise'].str.match(r'^\d{1,2}/\d{1,2}/\d{2,4}', na=False)]

# Remove very short exercise names (likely artifacts)
df_clean = df_clean[df_clean['exercise'].str.len() > 3]

# Remove rows starting with common note markers
df_clean = df_clean[~df_clean['exercise'].str.match(r'^[>+\-*:]', na=False)]

print(f"Rows after cleaning: {len(df_clean)} (removed {len(df) - len(df_clean)} rows)")
df_clean.head(10)

## View Exercises by Date

In [ ]:
# See exercises for a specific date
sample_date = df_clean['date'].dropna().iloc[0]
print(f"Exercises for {sample_date}:")
df_clean[df_clean['date'] == sample_date][['exercise', 'sets', 'reps', 'weight', 'weight_unit']]

## Top Exercises by Frequency

In [ ]:
# Most common exercises
exercise_counts = df_clean['exercise'].value_counts().head(30)
print("Top 30 Most Common Exercises:")
print(exercise_counts)

## Exercises with Weight Data

In [ ]:
# Exercises that have weight data
with_weight = df_clean[df_clean['weight'].notna()]
print(f"Exercises with weight data: {len(with_weight)}")
print("\nSample exercises with weight:")
with_weight[['date', 'exercise', 'weight', 'weight_unit']].head(20)

## Weight Progress Over Time (Example: Bicep Curls)

In [ ]:
# Find an exercise with multiple entries
exercise_name = 'Bicep Curls'  # Change this to explore other exercises

exercise_data = df_clean[df_clean['exercise'].str.contains(exercise_name, case=False, na=False)]
print(f"Found {len(exercise_data)} entries for exercises containing '{exercise_name}':")
exercise_data[['date', 'exercise', 'weight', 'weight_unit']].sort_values('date')

## Save Cleaned Data

In [ ]:
# Save the cleaned data
df_clean.to_csv('../data_samples/parsed_workouts_clean.csv', index=False)
print(f"Saved {len(df_clean)} rows to parsed_workouts_clean.csv")

## View Raw OCR Content (for debugging)

Use this to understand what the OCR extracted and improve the parser if needed.

In [ ]:
# Load and view raw OCR content from one file
ocr_file = '../data_samples/ocr_outputs/1-10.pdf.json'
with open(ocr_file, 'r') as f:
    ocr_data = json.load(f)

# Show first 3000 characters of content
content = ocr_data.get('content', '')
print("=== First 3000 chars of OCR content ===")
print(content[:3000])